# PPE / Safety-Helmet Detection with YOLOv8

Trains an object-detection model that finds **people and safety helmets** in workplace images.

**Run this in Google Colab** (free GPU): open https://colab.research.google.com , upload this notebook, then set the runtime to GPU via *Runtime -> Change runtime type -> T4 GPU*. Your MacBook Air has no NVIDIA GPU, so training locally would be very slow; Colab's free T4 trains this in a few minutes.

Pipeline: install -> get dataset -> train -> evaluate (mAP) -> run inference -> download `best.pt`.

In [ ]:
# 1. Confirm we have a GPU (should print a Tesla T4 or similar)
!nvidia-smi -L

In [ ]:
# 2. Install
!pip -q install ultralytics roboflow
import ultralytics; ultralytics.checks()

## 3. Get the dataset

We use a public **hard-hat / PPE detection** dataset from **Roboflow Universe** (classes typically: `head`, `helmet`, `person`). Roboflow exports it already in YOLOv8 format with a `data.yaml`.

**Steps:**
1. Make a free account at https://roboflow.com
2. Open a hard-hat / PPE dataset on Roboflow Universe (search "hard hat workers" or "PPE detection").
3. Click **Download this Dataset -> YOLOv8 -> show download code**. Roboflow gives you a snippet with *your* API key.
4. Paste that snippet in place of the cell below.

The snippet looks like this (replace with your own key, workspace, project, version):

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
project = rf.workspace("WORKSPACE").project("PROJECT")
dataset = project.version(VERSION).download("yolov8")

DATA_YAML = dataset.location + "/data.yaml"
print("data.yaml at:", DATA_YAML)

## 4. Train

YOLOv8n (nano) is small and fast, a good beginner default. Start with 30 epochs; raise if the loss is still falling.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # start from COCO-pretrained weights (transfer learning)
results = model.train(
    data=DATA_YAML,
    epochs=30,
    imgsz=640,
    batch=16,
    patience=10,          # early-stop if no improvement for 10 epochs
    project="ppe",
    name="yolov8n_ppe",
)

## 5. Evaluate

`mAP50` and `mAP50-95` are the standard object-detection metrics. Note the numbers the run prints - these are the honest figures to put on your resume.

In [ ]:
metrics = model.val()
print("mAP50-95:", round(float(metrics.box.map), 4))
print("mAP50:   ", round(float(metrics.box.map50), 4))
print("per-class mAP50-95:", {model.names[i]: round(float(v), 3) for i, v in enumerate(metrics.box.maps)})

## 6. Run inference on the validation images and look at the boxes

In [ ]:
import glob, os
from IPython.display import Image, display

val_dir = os.path.join(dataset.location, "valid", "images")
sample = sorted(glob.glob(val_dir + "/*.jpg"))[:3]
pred = model.predict(sample, conf=0.35, imgsz=640, save=True)
for p in glob.glob(pred[0].save_dir + "/*.jpg")[:3]:
    display(Image(filename=p, width=500))

## 7. Download your trained weights

`best.pt` is the model. Download it, then:
- copy it into the GitHub repo under `weights/best.pt` (for local `infer.py`), and
- upload it to your Hugging Face Space under `weights/best.pt` (for the live demo).

In [ ]:
from google.colab import files
best = model.trainer.best  # path to best.pt
print("best weights:", best)
files.download(str(best))